In [1]:
import os
import numpy as np
import pandas as pd

In [2]:
def listar_arquivos(pasta: str):
    if not os.path.exists(pasta):
        print(f"A pasta '{pasta}' não existe.")
        return

    arquivos = [f for f in os.listdir(pasta) if os.path.isfile(os.path.join(pasta, f))]
    lista_arquivos = []
    for arquivo in arquivos:
        lista_arquivos.append(arquivo)

    if lista_arquivos:
        return [arquivo for arquivo in lista_arquivos if 'landmarks' in arquivo]
    else:
        print(f"Nenhum arquivo encontrado na pasta '{pasta}'.")

In [3]:
# ---------- utilidades geométricas ----------
def angle_deg(a, b, c):
    """
    Ângulo ABC (em graus), dado A,B,C como arrays (N,2) ou (N,3).
    Retorna array (N,).
    """
    ba = a - b
    bc = c - b
    # produto escalar e norma
    dot = np.sum(ba * bc, axis=1)
    n1 = np.linalg.norm(ba, axis=1)
    n2 = np.linalg.norm(bc, axis=1)
    denom = (n1 * n2) + 1e-9
    cosang = np.clip(dot / denom, -1.0, 1.0)
    return np.degrees(np.arccos(cosang))


def rolling_smooth(x, win=7):
    # win ímpar é melhor
    win = max(3, int(win))
    if win % 2 == 0:
        win += 1
    return pd.Series(x).rolling(win, center=True, min_periods=1).mean().to_numpy()

In [4]:
# ---------- detecção simples de picos/vales (sem SciPy) ----------
def local_extrema(x, order=5, mode="max"):
    """
    Retorna índices de máximos/minimos locais.
    order = tamanho da vizinhança: o ponto deve ser o maior/menor dentro do intervalo [i-order, i+order]
    """
    x = np.asarray(x)
    idx = []
    n = len(x)
    for i in range(order, n - order):
        window = x[i - order : i + order + 1]
        if mode == "max":
            if x[i] == np.max(window):
                idx.append(i)
        else:
            if x[i] == np.min(window):
                idx.append(i)
    return np.array(idx, dtype=int)


def filter_by_distance(indices, min_distance):
    """
    Se tiver eventos muito próximos, mantém só um a cada min_distance (o primeiro).
    """
    if len(indices) == 0:
        return indices
    kept = [indices[0]]
    last = indices[0]
    for i in indices[1:]:
        if i - last >= min_distance:
            kept.append(i)
            last = i
    return np.array(kept, dtype=int)


def filter_by_prominence(x, indices, prominence=5.0, mode="max"):
    """
    Filtro simples de 'prominence': compara com vizinhos imediatos.
    Não é o prominence verdadeiro do SciPy, mas ajuda a cortar tremulações.
    """
    x = np.asarray(x)
    out = []
    for i in indices:
        if i <= 0 or i >= len(x) - 1:
            continue
        if mode == "max":
            if (x[i] - max(x[i-1], x[i+1])) >= prominence:
                out.append(i)
        else:
            if (min(x[i-1], x[i+1]) - x[i]) >= prominence:
                out.append(i)
    return np.array(out, dtype=int)

In [5]:
# ---------- leitura do CSV e escolha do lado ----------
def pick_side(frames_df):
    """
    Escolhe LEFT ou RIGHT baseado em visibilidade média dos landmarks de braço.
    """
    left_vis = []
    right_vis = []
    # ombro, cotovelo, punho
    for lm in [11, 13, 15]:
        left_vis.append(frames_df.get(f"lm{lm}_vis"))
    for lm in [12, 14, 16]:
        right_vis.append(frames_df.get(f"lm{lm}_vis"))

    left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
    right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)

    left_score = np.nanmean(left)
    right_score = np.nanmean(right)
    return "LEFT" if left_score >= right_score else "RIGHT"


def get_arm_points(frames_df, side="LEFT", use_xy=True):
    """
    Retorna arrays A=ombro, B=cotovelo, C=punho.
    use_xy=True -> usa x,y (mais robusto pra câmera 2D)
    """
    if side == "LEFT":
        sh, el, wr = 11, 13, 15
    else:
        sh, el, wr = 12, 14, 16

    if use_xy:
        A = frames_df[[f"lm{sh}_x", f"lm{sh}_y"]].to_numpy()
        B = frames_df[[f"lm{el}_x", f"lm{el}_y"]].to_numpy()
        C = frames_df[[f"lm{wr}_x", f"lm{wr}_y"]].to_numpy()
    else:
        A = frames_df[[f"lm{sh}_x", f"lm{sh}_y", f"lm{sh}_z"]].to_numpy()
        B = frames_df[[f"lm{el}_x", f"lm{el}_y", f"lm{el}_z"]].to_numpy()
        C = frames_df[[f"lm{wr}_x", f"lm{wr}_y", f"lm{wr}_z"]].to_numpy()

    vis = frames_df[[f"lm{sh}_vis", f"lm{el}_vis", f"lm{wr}_vis"]].to_numpy()
    vis_mean = np.nanmean(vis, axis=1)
    return A, B, C, vis_mean


def interpolate_small_gaps(frames_df, cols):
    """
    Interpola NaNs em colunas numéricas (bom para pequenos buracos de detecção).
    """
    df = frames_df.copy()
    for c in cols:
        df[c] = df[c].interpolate(limit_direction="both")
    return df


In [6]:
# ---------- pipeline: frames -> reps ----------
def build_reps_from_csv(
    csv_path,
    smooth_window=9,
    extrema_order=7,
    min_rep_seconds=0.6,
    min_rep_rom_deg=25.0,
    prominence_deg=3.0,
):
    frames_df = pd.read_csv(csv_path)

    # garantir colunas essenciais
    if "frame" not in frames_df.columns or "time_sec" not in frames_df.columns:
        raise ValueError("CSV precisa ter colunas 'frame' e 'time_sec'.")

    # interpolar coordenadas básicas para reduzir NaNs
    coord_cols = [c for c in frames_df.columns if c.endswith(("_x", "_y", "_z"))]
    frames_df = interpolate_small_gaps(frames_df, coord_cols)

    # escolher lado automaticamente
    side = pick_side(frames_df)

    # obter pontos do braço
    A, B, C, vis_mean = get_arm_points(frames_df, side=side, use_xy=True)

    # ângulo do cotovelo por frame
    elbow = angle_deg(A, B, C)

    # suavização
    elbow_s = rolling_smooth(elbow, win=smooth_window)

    # fps aproximado
    t = frames_df["time_sec"].to_numpy()
    dt = np.nanmedian(np.diff(t))
    fps = 1.0 / dt if dt and dt > 0 else 30.0
    min_distance = int(round(min_rep_seconds * fps))

    # extremos: no curl, geralmente:
    # - ângulo alto ~ braço estendido (topo do ângulo)
    # - ângulo baixo ~ braço flexionado (mínimo do ângulo)
    maxima = local_extrema(elbow_s, order=extrema_order, mode="max")
    minima = local_extrema(elbow_s, order=extrema_order, mode="min")

    # filtros básicos
    maxima = filter_by_distance(maxima, min_distance=min_distance)
    minima = filter_by_distance(minima, min_distance=min_distance)

    maxima = filter_by_prominence(elbow_s, maxima, prominence=prominence_deg, mode="max")
    minima = filter_by_prominence(elbow_s, minima, prominence=prominence_deg, mode="min")

    # montar reps: (max -> min -> max)
    reps = []
    max_set = set(maxima.tolist())
    min_set = set(minima.tolist())

    print("SIDE:", side)
    print("elbow_s stats:",
        "min", float(np.nanmin(elbow_s)),
        "max", float(np.nanmax(elbow_s)),
        "range", float(np.nanmax(elbow_s) - np.nanmin(elbow_s)))

    # Quantos extremos antes de filtros?
    maxima0 = local_extrema(elbow_s, order=extrema_order, mode="max")
    minima0 = local_extrema(elbow_s, order=extrema_order, mode="min")
    print("extrema raw - maxima:", len(maxima0), "minima:", len(minima0))

    # Depois de distância
    maxima1 = filter_by_distance(maxima0, min_distance=int(round(min_rep_seconds * fps)))
    minima1 = filter_by_distance(minima0, min_distance=int(round(min_rep_seconds * fps)))
    print("extrema dist - maxima:", len(maxima1), "minima:", len(minima1))

    # Se você estiver usando prominence, veja o impacto
    maxima2 = filter_by_prominence(elbow_s, maxima1, prominence=prominence_deg, mode="max")
    minima2 = filter_by_prominence(elbow_s, minima1, prominence=prominence_deg, mode="min")
    print("extrema prom - maxima:", len(maxima2), "minima:", len(minima2))


    # estratégia: percorre máximos e busca um mínimo entre eles
    for i in range(len(maxima) - 1):
        start = maxima[i]
        end = maxima[i + 1]
        if end - start < min_distance:
            continue

        # mínimos dentro do intervalo
        mins_between = minima[(minima > start) & (minima < end)]
        if len(mins_between) == 0:
            continue

        # pega o mínimo mais “profundo” (menor ângulo) dentro do intervalo
        peak = mins_between[np.argmin(elbow_s[mins_between])]

        # ROM (diferença entre ângulo estendido e flexionado)
        rom = elbow_s[start] - elbow_s[peak]
        if np.isnan(rom) or rom < min_rep_rom_deg:
            continue

        reps.append({
            "rep_id": len(reps),
            "side": side,
            "start_idx": int(start),
            "peak_idx": int(peak),   # flexão máxima (menor ângulo)
            "end_idx": int(end),
            "start_frame": int(frames_df.loc[start, "frame"]),
            "peak_frame": int(frames_df.loc[peak, "frame"]),
            "end_frame": int(frames_df.loc[end, "frame"]),
            "start_time": float(frames_df.loc[start, "time_sec"]),
            "peak_time": float(frames_df.loc[peak, "time_sec"]),
            "end_time": float(frames_df.loc[end, "time_sec"]),
            "rep_duration": float(frames_df.loc[end, "time_sec"] - frames_df.loc[start, "time_sec"]),
            "rom_deg": float(rom),
            "elbow_start_deg": float(elbow_s[start]),
            "elbow_peak_deg": float(elbow_s[peak]),
            "mean_vis_arm": float(np.nanmean(vis_mean[start:end+1])),
        })

    reps_df = pd.DataFrame(reps)

    # também devolve o frames_df com o sinal guia (útil para plot/depurar)
    frames_out = frames_df.copy()
    frames_out["elbow_deg_raw"] = elbow
    frames_out["elbow_deg_smooth"] = elbow_s
    frames_out["arm_vis_mean"] = vis_mean

    return frames_out, reps_df

In [ ]:
# ---------- exemplo de uso ----------
if __name__ == "__main__":
    
    raiz = r"./saida/landmarks/"

    for arquivo in listar_arquivos(raiz):
        csv_path = os.path.join(raiz, arquivo)
        print(csv_path)
        frames_df, reps_df = build_reps_from_csv(
            csv_path,
            smooth_window=11,
            extrema_order=7,
            min_rep_seconds=0.7,
            min_rep_rom_deg=25.0,
            prominence_deg=0,
        )

        print("Reps detectadas:", len(reps_df))
        print(reps_df.head())

        # Salvar para revisar/corrigir manualmente se quiser
        reps_df.to_csv(f"{raiz.replace('landmarks/', 'feature_extraction/')}{arquivo.replace('_landmarks.csv', '_reps.csv')}", index=False)


./saida/landmarks/aug_brightness_friends_record_01_landmarks.csv
SIDE: RIGHT
elbow_s stats: min 4.08136569936849 max 178.826004904449 range 174.74463920508052
extrema raw - maxima: 61 minima: 58
extrema dist - maxima: 50 minima: 46
extrema prom - maxima: 50 minima: 46
Reps detectadas: 36
   rep_id   side  start_idx  peak_idx  end_idx  start_frame  peak_frame  \
0       0  RIGHT         56        74       78           56          74   
1       1  RIGHT        101       125      143          101         125   
2       2  RIGHT        143       151      172          143         151   
3       3  RIGHT        172       179      200          172         179   
4       4  RIGHT        200       242      246          200         242   

   end_frame  start_time  peak_time  end_time  rep_duration     rom_deg  \
0         78    1.866667   2.466667  2.600000      0.733333   54.370219   
1        143    3.366667   4.166667  4.766667      1.400000  155.541512   
2        172    4.766667   5.033333

C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: RIGHT
elbow_s stats: min 54.79209769577185 max 167.25139760988415 range 112.45929991411231
extrema raw - maxima: 47 minima: 50
extrema dist - maxima: 23 minima: 25
extrema prom - maxima: 23 minima: 25
Reps detectadas: 20
   rep_id   side  start_idx  peak_idx  end_idx  start_frame  peak_frame  \
0       0  RIGHT         28        55       84           28          55   
1       1  RIGHT         84       141      171           84         141   
2       2  RIGHT        171       199      228          171         199   
3       3  RIGHT        228       256      286          228         256   
4       4  RIGHT        286       313      342          286         313   

   end_frame  start_time  peak_time   end_time  rep_duration    rom_deg  \
0         84    0.933333   1.833333   2.800000      1.866667  45.463634   
1        171    2.800000   4.700000   5.700000      2.900000  84.029311   
2        228    5.700000   6.633333   7.600000      1.900000  86.345252   
3        286    7.6000

C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: RIGHT
elbow_s stats: min 4.4579101689805745 max 176.51137282657575 range 172.0534626575952
extrema raw - maxima: 12 minima: 13
extrema dist - maxima: 12 minima: 13
extrema prom - maxima: 12 minima: 13
Reps detectadas: 11
   rep_id   side  start_idx  peak_idx  end_idx  start_frame  peak_frame  \
0       0  RIGHT         54        76      100           54          76   
1       1  RIGHT        100       121      147          100         121   
2       2  RIGHT        147       162      194          147         162   
3       3  RIGHT        194       221      241          194         221   
4       4  RIGHT        241       260      288          241         260   

   end_frame  start_time  peak_time  end_time  rep_duration     rom_deg  \
0        100    1.800000   2.533333  3.333333      1.533333  151.972360   
1        147    3.333333   4.033333  4.900000      1.566667  157.222520   
2        194    4.900000   5.400000  6.466667      1.566667  137.448228   
3        241    6.4666

...

SIDE: RIGHT
elbow_s stats: min 11.888726260338872 max 175.27539563143785 range 163.38666937109898
extrema raw - maxima: 29 minima: 28
extrema dist - maxima: 24 minima: 25
extrema prom - maxima: 24 minima: 25
Reps detectadas: 20
   rep_id   side  start_idx  peak_idx  end_idx  start_frame  peak_frame  \
0       0  RIGHT          7        29       44            7          29   
1       1  RIGHT         73       143      172           73         143   
2       2  RIGHT        172       199      227          172         199   
3       3  RIGHT        227       256      286          227         256   
4       4  RIGHT        286       312      343          286         312   

   end_frame  start_time  peak_time   end_time  rep_duration    rom_deg  \
0         44    0.233333   0.966667   1.466667      1.233333  30.788610   
1        172    2.433333   4.766667   5.733333      3.300000  92.186771   
2        227    5.733333   6.633333   7.566667      1.833333  86.464477   
3        286    7.566

C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


SIDE: LEFT
elbow_s stats: min 2.8170852294393987 max 171.78230997034333 range 168.96522474090395
extrema raw - maxima: 6 minima: 7
extrema dist - maxima: 5 minima: 4
extrema prom - maxima: 5 minima: 4
Reps detectadas: 3
   rep_id  side  start_idx  peak_idx  end_idx  start_frame  peak_frame  \
0       0  LEFT        111       161      220          111         161   
1       1  LEFT        220       287      297          220         287   
2       2  LEFT        350       413      423          350         413   

   end_frame  start_time  peak_time   end_time  rep_duration     rom_deg  \
0        220    3.700000   5.366667   7.333333      3.633333  167.011470   
1        297    7.333333   9.566667   9.900000      2.566667  164.227807   
2        423   11.666667  13.766667  14.100000      2.433333  163.544093   

   elbow_start_deg  elbow_peak_deg  mean_vis_arm  
0       171.782310        4.770840      0.997059  
1       171.756047        7.528240      0.996449  
2       171.028076       

C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:14: RuntimeWarning: Mean of empty slice
  left = np.nanmean(np.vstack([s.to_numpy() for s in left_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:15: RuntimeWarning: Mean of empty slice
  right = np.nanmean(np.vstack([s.to_numpy() for s in right_vis if s is not None]), axis=0)
C:\Users\Gush\AppData\Local\Temp\ipykernel_14460\3443660567.py:42: RuntimeWarning: Mean of empty slice
  vis_mean = np.nanmean(vis, axis=1)


: 